In [1]:
import os
import pandas as pd
import numpy as np
from datetime import date

os.chdir('/Users/anasantana/Desktop/copa')

In [2]:
results   = pd.read_csv('results.csv',      parse_dates=['date'])
shootouts = pd.read_csv('shootouts.csv',    parse_dates=['date'])
fnames    = pd.read_csv('former_names.csv', parse_dates=['start_date', 'end_date'])

print(f'Results:   {len(results):,} matches')
print(f'Shootouts: {len(shootouts):,} matches')
print(f'Name mappings: {len(fnames)}')

Results:   49,477 matches
Shootouts: 678 matches
Name mappings: 36


In [3]:
name_map = dict(zip(fnames['former'], fnames['current']))
def normalize_team(name): return name_map.get(name, name)

for col in ['home_team', 'away_team']:
    results[col]   = results[col].map(normalize_team)
    shootouts[col] = shootouts[col].map(normalize_team)
shootouts['winner'] = shootouts['winner'].map(normalize_team)

print('Names normalized.')

Names normalized.


In [4]:
shootouts_slim = shootouts[['date','home_team','away_team','winner']].rename(columns={'winner':'shootout_winner'})
df = results.merge(shootouts_slim, on=['date','home_team','away_team'], how='left')

def true_winner(row):
    if pd.notna(row['shootout_winner']): return row['shootout_winner']
    elif row['home_score'] > row['away_score']: return row['home_team']
    elif row['away_score'] > row['home_score']: return row['away_team']
    else: return 'Draw'

df['winner'] = df.apply(true_winner, axis=1)
print(f'Total matches after merge: {len(df):,}')

Total matches after merge: 49,477


In [5]:
wc2026_matches = df[(df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)]
WC2026_TEAMS   = set(wc2026_matches['home_team'].tolist() + wc2026_matches['away_team'].tolist())

print(f'WC 2026 teams: {len(WC2026_TEAMS)}')
print(sorted(WC2026_TEAMS))

WC 2026 teams: 48
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']


## 5. Three-way split

- **train** — all history before 2023, WC 2026 teams only. Core training set.
- **historical_test** — 2023–2025 matches, WC 2026 teams only. Lets us validate accuracy on recent games before touching the tournament.
- **wc_test** — WC 2026 games only. The main backtest.

Keeping `historical_test` separate means we never accidentally train on games
we want to use for validation.

In [6]:
both_wc   = df['home_team'].isin(WC2026_TEAMS) & df['away_team'].isin(WC2026_TEAMS)
is_wc2026 = (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)

train            = df[both_wc & (df['date'] < '2023-01-01') & ~is_wc2026].copy()
historical_test  = df[both_wc & (df['date'] >= '2023-01-01') & ~is_wc2026].copy()
wc_test          = df[is_wc2026].copy()

for d in [train, historical_test, wc_test]:
    d.sort_values('date', inplace=True)
    d.reset_index(drop=True, inplace=True)

print(f'train:           {len(train):,} matches  ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'historical_test: {len(historical_test):,} matches  ({historical_test["date"].min().date()} → {historical_test["date"].max().date()})')
print(f'wc_test:         {len(wc_test):,} matches  ({wc_test["date"].min().date()} → {wc_test["date"].max().date()})')

train:           7,124 matches  (1872-11-30 → 2022-12-18)
historical_test: 403 matches  (2023-01-09 → 2026-06-09)
wc_test:         72 matches  (2026-06-11 → 2026-06-27)


In [8]:
TODAY = pd.Timestamp(date.today())

def recency_weight(d):
    days = (TODAY - d).days
    if days <= 365*3:  return 3.0
    elif days <= 365*8: return 2.0
    else: return 1.0

MAJOR = {'FIFA World Cup','Copa América','UEFA Euro','Africa Cup of Nations','AFC Asian Cup','CONCACAF Gold Cup'}
def t_weight(t):
    if t in MAJOR: return 2.0
    elif 'qualifier' in t.lower() or 'qualification' in t.lower(): return 1.5
    else: return 1.0

train['recency_weight']    = train['date'].apply(recency_weight)
train['tournament_weight'] = train['tournament'].apply(t_weight)
train['weight']            = train['recency_weight'] * train['tournament_weight']

print('Weights applied to training data.')
print(f'Weight range: {train["weight"].min()} – {train["weight"].max()}')

Weights applied to training data.
Weight range: 1.0 – 4.0


In [10]:
def head_to_head(team_a, team_b):
    mask = (
        ((train['home_team'] == team_a) & (train['away_team'] == team_b)) |
        ((train['home_team'] == team_b) & (train['away_team'] == team_a))
    )
    return train[mask][['date', 'home_team', 'away_team', 'home_score',
                         'away_score', 'tournament', 'winner', 'weight']]

h2h = head_to_head('Brazil', 'Mexico')
print(f'Brazil vs Mexico — all time (training): {len(h2h)} matches')
h2h

Brazil vs Mexico — all time (training): 41 matches


,date,home_team,away_team,home_score,away_score,tournament,winner,weight
1015,1950-06-24,Brazil,Mexico,4.0,0.0,FIFA World Cup,Brazil,2.0
1071,1952-04-06,Brazil,Mexico,2.0,0.0,Pan American Championship,Brazil,1.0
1161,1954-06-16,Brazil,Mexico,5.0,0.0,FIFA World Cup,Brazil,2.0
1229,1956-03-08,Mexico,Brazil,1.0,2.0,Pan American Championship,Brazil,1.0
1411,1960-03-06,Brazil,Mexico,2.0,2.0,Pan American Championship,Draw,1.0
1414,1960-03-15,Brazil,Mexico,2.0,1.0,Pan American Championship,Brazil,1.0
1523,1962-05-30,Brazil,Mexico,2.0,0.0,FIFA World Cup,Brazil,2.0
1814,1968-07-07,Mexico,Brazil,0.0,2.0,Friendly,Brazil,1.0
1815,1968-07-10,Mexico,Brazil,2.0,1.0,Friendly,Mexico,1.0
1833,1968-10-31,Brazil,Mexico,1.0,2.0,Friendly,Mexico,1.0
